# Robust Hadamard via `VariationalRolloutProblem`

Solves for a single-qubit Hadamard gate that is **first-order robust** to perturbations along {X, Y, Z} simultaneously, using the new indirect (rollout) variational template.

**Why indirect vs direct?** The direct `VariationalSplinePulseProblem` carries the augmented state `[Ũ⃗; ∂Ũ⃗_X; ∂Ũ⃗_Y; ∂Ũ⃗_Z]` (dim = 32) as NLP decision variables, with a dense matrix-exp Jacobian per knot transition (cost ∝ `(iso_dim·(1+n_vars))³ = 32³`). The indirect version (this notebook) drops `:var_Ũ⃗` from the NLP, computing `‖∂Ũ⃗(T)‖²` via forward rollout of the variational ODE and getting gradients via ForwardDiff through the rollout.

**What this notebook does:**
1. Set up a 2-level system with X/Y drives, target = Hadamard
2. Define 3 variational error directions: H_X, H_Y, H_Z
3. Solve with `VariationalRolloutProblem` (Q_r > 0 → robust)
4. Solve same problem with `Q_r = 0` (non-robust baseline)
5. Verify robustness: sweep `ε ∈ [-0.1, 0.1]` for each error direction, plot 1 − F

Robust curves should be flatter near `ε = 0` than non-robust curves, and the parabolic minimum should be visibly broader.

In [ ]:
import Pkg
Pkg.activate(@__DIR__)
piccolo_path       = joinpath(@__DIR__, "..", "..", "..", "Piccolo.jl")
directtrajopt_path = joinpath(@__DIR__, "..", "..", "..", "DirectTrajOpt.jl")
Pkg.develop([
    Pkg.PackageSpec(path = piccolo_path),
    Pkg.PackageSpec(path = directtrajopt_path),
])
Pkg.add(["CairoMakie", "Ipopt", "FFTW", "JLD2"])
Pkg.instantiate()

using Piccolo
using LinearAlgebra
using Random
using Printf
using CairoMakie

Random.seed!(42)

## 1. Build the variational system

- Hamiltonian: `H(u) = uₓ · X + u_y · Y`  (no drift; we want Hadamard via XY drives only)
- Drive bounds: ±2π·0.5 rad/ns ≈ ±500 MHz amplitude  
- Variational directions: `H_X, H_Y, H_Z` (X, Y, and Z error channels)

In [ ]:
# Time-independent setup: zero drift, X and Y as drives
const H_drift = 0.0 * PAULIS.Z
const H_drives = [PAULIS.X, PAULIS.Y]
const H_vars   = [PAULIS.X, PAULIS.Y, PAULIS.Z]

const a_bound = 2π * 0.5   # rad/ns
const drive_bounds = fill(a_bound, length(H_drives))

varsys = VariationalQuantumSystem(H_drift, H_drives, H_vars, drive_bounds)

println("varsys.levels    = ", varsys.levels)
println("varsys.n_drives  = ", varsys.n_drives)
println("length(G_vars)   = ", length(varsys.G_vars), "  (X, Y, Z)")
println("varsys.time_dep  = ", varsys.time_dependent)

## 2. Trajectory and target

Gate time T = 2 ns; N = 15 knots (Δt ≈ 0.143 ns). Cubic Hermite spline with the tangents `:du` as additional NLP variables. Initial pulse: small random.

In [ ]:
const T_gate = 2.0      # ns
const N_knots = 15

times = collect(range(0.0, T_gate, length = N_knots))
u_init  = 0.1 .* randn(varsys.n_drives, N_knots)
du_init = 0.1 .* randn(varsys.n_drives, N_knots)
pulse = CubicSplinePulse(u_init, du_init, times)

const U_goal = GATES.H
println("Target unitary (Hadamard):")
display(U_goal)

## 3. Build and solve the robust problem (`Q_r = 1.0`)

Knobs:
- `Q = 200.0` — fidelity weight
- `Q_r = 1.0` — robustness weight (`‖∂Ũ⃗_i(T)‖²` for each error channel `i ∈ {X,Y,Z}`)
- `R = 1e-3` — control regularization (`R_u`, `R_du`)
- `du_bound = 5·a_bound` — keep Hermite tangent magnitudes finite (else the cubic spline can overshoot between knots)
- `timesteps_all_equal = true` — fixed Δt (otherwise NLP would also tune Δt_k)

In [ ]:
qcp_robust = VariationalRolloutProblem(
    varsys, pulse, U_goal, N_knots;
    Q     = 200.0,
    Q_r   = 1.0,
    R     = 1e-3,
    du_bound = 5 * a_bound,
    piccolo_options = PiccoloOptions(
        verbose = true,
        timesteps_all_equal = true,
    ),
)

@time solve!(
    qcp_robust;
    max_iter = 500,
    options = IpoptOptions(eval_hessian = false, print_level = 4),
)

## 4. Build and solve the non-robust baseline (`Q_r = 0`)

Same setup, just `Q_r = 0` — the variational objective is silent. Pure infidelity minimization. Provides a reference for what fidelity-only optimization gives, so we can see the robustness improvement.

In [ ]:
# Re-seed so both problems start from the same initial pulse
Random.seed!(42)
u_init_2  = 0.1 .* randn(varsys.n_drives, N_knots)
du_init_2 = 0.1 .* randn(varsys.n_drives, N_knots)
pulse_2 = CubicSplinePulse(u_init_2, du_init_2, times)

qcp_nonrobust = VariationalRolloutProblem(
    varsys, pulse_2, U_goal, N_knots;
    Q     = 200.0,
    Q_r   = 0.0,    # ← THE difference
    R     = 1e-3,
    du_bound = 5 * a_bound,
    piccolo_options = PiccoloOptions(
        verbose = false,
        timesteps_all_equal = true,
    ),
)

@time solve!(
    qcp_nonrobust;
    max_iter = 500,
    options = IpoptOptions(eval_hessian = false, print_level = 4),
)

## 5. Plot the optimized controls

Two pulses, side by side. We'll plot the cubic-Hermite-interpolated continuous control between knots, not just the knot values.

In [ ]:
function eval_cubic_hermite(u_knots::AbstractMatrix, du_knots::AbstractMatrix, ts::AbstractVector, t_query)
    # u_knots, du_knots: n_drives × N
    # ts: knot times
    # t_query: scalar
    n_drives, N = size(u_knots)
    if t_query <= ts[1]
        return u_knots[:, 1]
    elseif t_query >= ts[end]
        return u_knots[:, end]
    end
    k = searchsortedlast(ts, t_query)
    Δt = ts[k+1] - ts[k]
    τ  = (t_query - ts[k]) / Δt
    h00 = 2τ^3 - 3τ^2 + 1
    h10 =  τ^3 - 2τ^2 + τ
    h01 = -2τ^3 + 3τ^2
    h11 =  τ^3 -  τ^2
    return h00 * u_knots[:, k] + (h10 * Δt) * du_knots[:, k] +
           h01 * u_knots[:, k+1] + (h11 * Δt) * du_knots[:, k+1]
end

traj_robust    = get_trajectory(qcp_robust)
traj_nonrobust = get_trajectory(qcp_nonrobust)

t_fine = collect(range(0, T_gate, length=400))
u_robust_fine    = hcat([eval_cubic_hermite(traj_robust[:u],    traj_robust[:du],    times, t) for t in t_fine]...)
u_nonrobust_fine = hcat([eval_cubic_hermite(traj_nonrobust[:u], traj_nonrobust[:du], times, t) for t in t_fine]...)

fig = Figure(size=(1100, 500), fontsize=18)
ax1 = Axis(fig[1, 1], xlabel="t  [ns]", ylabel="u  [rad/ns]", title="Robust  (Q_r = 1.0)")
lines!(ax1, t_fine, u_robust_fine[1, :]; color=:crimson,     linewidth=2, label="u_X")
lines!(ax1, t_fine, u_robust_fine[2, :]; color=:forestgreen, linewidth=2, label="u_Y")
axislegend(ax1; position=:rt)

ax2 = Axis(fig[1, 2], xlabel="t  [ns]", ylabel="u  [rad/ns]", title="Non-robust  (Q_r = 0)")
lines!(ax2, t_fine, u_nonrobust_fine[1, :]; color=:crimson,     linewidth=2, label="u_X")
lines!(ax2, t_fine, u_nonrobust_fine[2, :]; color=:forestgreen, linewidth=2, label="u_Y")
axislegend(ax2; position=:rt)

linkyaxes!(ax1, ax2)
display(fig)

## 6. Verify final fidelities at `ε = 0`

Both should be close to 1 — robustness is *first-order*, so the unperturbed fidelity isn't sacrificed in exchange. (If it were, you'd reduce `Q_r` or raise `Q`.)

In [ ]:
function final_unitary(traj)
    Ũ_f = traj[:Ũ⃗][:, end]
    return iso_vec_to_operator(Ũ_f)
end

function unitary_fidelity_to(U, U_target)
    d = size(U_target, 1)
    return abs2(tr(U_target' * U)) / d^2
end

U_robust    = final_unitary(traj_robust)
U_nonrobust = final_unitary(traj_nonrobust)

F_robust    = unitary_fidelity_to(U_robust,    U_goal)
F_nonrobust = unitary_fidelity_to(U_nonrobust, U_goal)

@printf("Robust    (Q_r=1.0):  F(ε=0) = %.8f  (1−F = %.3e)\n", F_robust,    1 - F_robust)
@printf("Non-robust (Q_r=0 ):  F(ε=0) = %.8f  (1−F = %.3e)\n", F_nonrobust, 1 - F_nonrobust)

## 7. Susceptibility sweep — 1 − F vs ε for each error direction

For each error direction `H_err ∈ {X, Y, Z}` and ε on a grid:
1. Build a perturbed `QuantumSystem` with `H = H_drives · u + ε · H_err`
2. Propagate the optimized controls through that perturbed system (using `unitary_rollout_fidelity` with cubic Hermite interpolation, matching the NLP dynamics)
3. Record `1 − F`

Expected: the robust curves should be **quadratic-only** at ε ≈ 0 (no linear slope) for each direction, since the variational objective drives `∂U/∂ε = 0` at the design point. The non-robust curves can have any first-order behavior.

In [ ]:
function perturbed_fidelity(traj, H_err, ε)
    # Build a QuantumSystem with drift = ε·H_err
    sys_ε = QuantumSystem(ε * H_err, H_drives, drive_bounds)
    return unitary_rollout_fidelity(traj, sys_ε; interpolation = :cubic_hermite)
end

const εs = collect(range(-0.1, 0.1, length = 41))

error_directions = [
    ("X", PAULIS.X),
    ("Y", PAULIS.Y),
    ("Z", PAULIS.Z),
]

F_robust_vs_eps    = Dict{String, Vector{Float64}}()
F_nonrobust_vs_eps = Dict{String, Vector{Float64}}()

for (name, H_err) in error_directions
    F_arr_r  = Float64[]
    F_arr_nr = Float64[]
    for ε in εs
        push!(F_arr_r,  perturbed_fidelity(traj_robust,    H_err, ε))
        push!(F_arr_nr, perturbed_fidelity(traj_nonrobust, H_err, ε))
    end
    F_robust_vs_eps[name]    = F_arr_r
    F_nonrobust_vs_eps[name] = F_arr_nr
    @printf("  %s done — min F (robust): %.6f,  min F (non-robust): %.6f\n",
        name, minimum(F_arr_r), minimum(F_arr_nr))
end

## 8. Plot susceptibility curves

Three panels (one per error direction), log-y, comparing robust (red) vs non-robust (black). Robust curves should sit below non-robust near `ε = 0` and stay much flatter.

In [ ]:
fig = Figure(size = (1500, 500), fontsize = 20)
for (col, (name, _)) in enumerate(error_directions)
    ax = Axis(fig[1, col];
        xlabel = "ε",
        ylabel = "1 − F",
        title  = "Perturbation: $name",
        yscale = log10,
    )
    lines!(ax, εs, max.(1 .- F_robust_vs_eps[name],    1e-16);
        color = :crimson, linewidth = 2.5, label = "robust (Q_r=1.0)")
    lines!(ax, εs, max.(1 .- F_nonrobust_vs_eps[name], 1e-16);
        color = :black,   linewidth = 2.5, label = "non-robust (Q_r=0)")
    ylims!(ax, 1e-12, 1e-1)
    col == 1 && axislegend(ax; position = :lt, labelsize = 14)
end
Label(fig[0, :], "Hadamard robustness — VariationalRolloutProblem", fontsize = 22)
display(fig)

save(joinpath(@__DIR__, "robust_hadamard_variational_rollout_susceptibility.png"), fig)
println("Saved figure.")

## 9. What to expect

**At ε = 0**: both robust and non-robust should reach `F > 0.9999` (since `Q = 200` is much larger than `Q_r = 1`).

**Near ε = 0**: 
- Robust curves should be **quadratic from below** (no linear slope): `1 − F ≈ c·ε²` with small `c`.
- Non-robust curves can have either quadratic or linear-ish behavior (whichever the optimizer's free pulse happened to produce). Usually visibly steeper near zero.

**At larger ε** (say ε ≈ 0.05): the robust pulse should outperform the non-robust by a factor of 10× or more in infidelity. If they're indistinguishable, suspect:
- `Q_r` too small relative to `Q` — try `Q_r = 10` or larger
- `R` regularizers too large — they constrain the pulse from doing what the variational gradient wants
- Insufficient iterations — Ipopt should have converged (no `r` line-search restoration markers)

**Sanity check on `Q_r` scaling**: from `objectives.jl`, the rollout penalty is `Q_r · scale^4 · ‖∂Ũ⃗ᵢ(T)/scale‖² / (d · T² · n_vars)`. For our setup: `d = 2, T = 2, n_vars = 3`, so the divisor is `24`. To match `Q = 200`'s effective scale on something of order unity, `Q_r ≈ 200 · 24 / (typical ‖∂Ũ⃗‖²)` — but typical `‖∂Ũ⃗‖²` is hard to estimate a priori; sweep `Q_r ∈ {0.1, 1.0, 10, 100}` if the result isn't strong enough.